In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/03/04 23:23:34 WARN Utils: Your hostname, codespaces-b40668 resolves to a loopback address: 127.0.0.1; using 10.0.15.140 instead (on interface eth0)
26/03/04 23:23:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 23:23:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'3.5.3'

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-04 23:26:23--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.167.84.127, 3.167.84.228, 3.167.84.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.167.84.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   216MB/s    in 0.3s    

2026-03-04 23:26:23 (216 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [6]:
from pyspark.sql import types

In [10]:
df_check = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df_check.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [12]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [13]:
df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .parquet("yellow_tripdata_2025-11.parquet")

output_path = f'hw_yellow_2025-11/'

df_yellow \
    .repartition(4) \
    .write.parquet(output_path)

In [15]:
df_yellow.createOrReplaceTempView('df_yellow')

In [18]:
spark.sql("""
SELECT
    count(VendorID)
FROM
    df_yellow
WHERE  
    CAST(tpep_pickup_datetime AS DATE) = '2025-11-15'
""").show()

+---------------+
|count(VendorID)|
+---------------+
|         162604|
+---------------+



In [22]:
# Difference in hours
spark.sql("""
SELECT
    MAX(TIMESTAMPDIFF(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))
FROM
    df_yellow
""").show()

[Stage 17:=============================>                            (1 + 1) / 2]

+---------------------------------------------------------------------+
|max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))|
+---------------------------------------------------------------------+
|                                                                   90|
+---------------------------------------------------------------------+



In [23]:
spark.sql("""
SELECT
    MAX((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600.0) AS max_hours
FROM
    df_yellow
""").show()

[Stage 20:>                                                         (0 + 2) / 2]

+---------+
|max_hours|
+---------+
|90.646667|
+---------+



In [19]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|         186|           1|       14.9|  0.0|    0.5|       1.5|         0.0|                  1.0

In [29]:
!wget -O hw_taxi_zone.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-05 00:05:11--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.167.84.228, 3.167.84.86, 3.167.84.127, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.167.84.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘hw_taxi_zone.csv’

hw_taxi_zone.csv    100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-05 00:05:11 (272 MB/s) - ‘hw_taxi_zone.csv’ saved [12331/12331]



In [30]:
df = spark.read \
    .option("header", "true") \
    .csv('hw_taxi_zone.csv')

In [32]:
df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [33]:
df.createOrReplaceTempView('df_lookup')

In [40]:
spark.sql("""
SELECT
    y.PULocationID, d.Zone,
    COUNT(y.PULocationID) AS min_PU_loc
FROM
    df_yellow y 
    JOIN df_lookup d ON y.PULocationID = d.LocationID
    GROUP BY 1,2
    ORDER BY 3 LIMIT 5
""").show()

[Stage 37:>                                                         (0 + 2) / 2]

+------------+--------------------+----------+
|PULocationID|                Zone|min_PU_loc|
+------------+--------------------+----------+
|           5|       Arden Heights|         1|
|          84|Eltingville/Annad...|         1|
|         105|Governor's Island...|         1|
|         187|       Port Richmond|         3|
|         111| Green-Wood Cemetery|         4|
+------------+--------------------+----------+

